# 2 · Evaluating a strategy

Is my strategy any good, and how would I know?

The short answer is that a single run cannot tell you. This notebook builds
up to the comparison that can.

In [1]:
import tradefloor as tf

universe = tf.Universe.random(40, seed=111)
print("universe:", len(universe), "instruments")

universe: 40 instruments


## Writing a strategy down

`StrategySpec` records a strategy as a declarative, versioned, hashable
document rather than a Python callable. The reason is citation: a reader can
re-run your seed and get your market, but there is no way to hand them a
callable. A spec they can.

The grammar is small on purpose. Five behavioural parts: a signal, a
concentration (`top_k`), an exposure (`gross`), a participation cap, and
`execution.cadence` -- how often the strategy re-decides. Cadence is in
the spec rather than in the harness because it moves results more than any
signal parameter, so two runs of one fingerprint that differed in it could
disagree in sign. You can see all five in the JSON below.

In [2]:
spec = tf.StrategySpec.momentum(lookback_days=1.0, top_k=5)

print(spec.to_json())
print("fingerprint:", spec.fingerprint)

{
  "spec_version": 1,
  "signal": {
    "kind": "momentum",
    "lookback_days": 1.0
  },
  "portfolio": {
    "gross": 1.0,
    "top_k": 5
  },
  "execution": {
    "cadence": "step",
    "max_participation": 0.02
  },
  "seed": null
}
fingerprint: e6bbc35c6f0968b1f178e1f7ee926d449a8d3fa72440e476dd8b25c7a6a50895


It cannot express path dependence (stop losses, drawdown limits, anything
reading its own P&L history), conditional logic, or custom signals. Those
need a Python agent, which works everywhere a spec does. The cost is that
the result has to cite code at a commit instead of a fingerprint.

## Running an evaluation

`evaluate` runs every entrant against an identical market, which makes the
comparison exact. It is still only one market, which is the limitation the
rest of this notebook deals with.

In [3]:
entrants = {"mine": spec}
entrants.update(tf.baselines.reference_agents(seed=7))

scores = tf.evaluate(entrants, seed=7, universe=universe, days=10)

print(f"{'agent':16s} {'return':>9s} {'trades':>7s} {'impact bps':>11s}")
for name, s in sorted(scores.items(), key=lambda kv: -kv[1].return_pct):
    print(f"{name:16s} {s.return_pct:8.3f}% {s.trades:7d} {s.impact_bps:11.2f}")

agent               return  trades  impact bps
oracle              8.967%     574       89.31
mean_reversion      7.769%     792        6.65
random             -1.189%    2386        6.25
buy_and_hold       -1.305%      40        6.32
mine               -2.638%     720        3.34
momentum           -2.638%     720        3.34


The baselines are included deliberately: a return means nothing until you
know what buy-and-hold did on the same market. Here `mean_reversion` is up
8.870% while `buy_and_hold` is down 1.362%, so that edge belongs to the
strategy rather than to the market.

## Capture ratio

`oracle` reads the simulator's own fair value, so it sees what no real
trader could. Capture ratio expresses P&L as a fraction of what `oracle`
earned. That is a reference point rather than a ceiling: `oracle` gets the
same gross exposure and participation cap as every other entrant and spends
them on a naive equal-weight rule, so a strategy with a better portfolio
under the same constraint can score above 1.0.

In [4]:
capture = tf.capture_ratio(scores)
for name, value in sorted(capture.items(), key=lambda kv: -kv[1]):
    print(f"  {name:16s} {value:7.3f}")

  mean_reversion     0.866
  random            -0.133
  buy_and_hold      -0.146
  mine              -0.294
  momentum          -0.294


1.0 is the reference's own result, a scale to read against rather than a
target.

## Why one seed isn't enough

The same comparison on three different market draws:

In [5]:
for seed in (7, 8, 9):
    e = {"mine": spec}
    e.update(tf.baselines.reference_agents(seed=seed))
    s = tf.evaluate(e, seed=seed, universe=universe, days=10)
    ranked = sorted(s.items(), key=lambda kv: -kv[1].return_pct)
    print(f"seed {seed}: " + "  ".join(f"{n}({v.return_pct:+.1f}%)"
                                        for n, v in ranked[:4]))

seed 7: oracle(+9.0%)  mean_reversion(+7.8%)  random(-1.2%)  buy_and_hold(-1.3%)


seed 8: mean_reversion(+5.6%)  oracle(+3.9%)  mine(+0.6%)  momentum(+0.6%)


seed 9: oracle(+9.2%)  mean_reversion(+6.9%)  random(-0.4%)  mine(-2.0%)


`oracle` and `mean_reversion` hold the top two places on all three draws, but
the places below them change hands, and `mine` lands anywhere from -0.2% to
-4.3% depending on which market it drew. Where the ordering changes between
seeds, a single-seed leaderboard is telling you about the seed rather than
the strategies.

## Ranking across seeds

`rank` runs many seeds and compares entrants pairwise on the same market
draw. Pairing removes the market from the comparison, so a modest number of
seeds is still informative.

It takes a factory rather than built agents. Agents are stateful, and a
reused instance carries one market's history into the next with no visible
symptom. A spec avoids this because it is rebuilt for each seed.

In [6]:
def make_agents():
    e = {"mine": tf.StrategySpec.momentum(lookback_days=1.0, top_k=5)}
    e.update(tf.baselines.reference_agents(seed=0))
    return e

ranking = tf.rank(make_agents, seeds=[1, 2, 3, 4, 5, 6],
                  universe=universe, days=5)

print(f"{'agent':16s} {'pooled capture':>15s} {'median P&L':>13s} {'first on':>9s}")
for r in ranking.table():
    pooled = "n/a" if r.pooled_capture is None else f"{r.pooled_capture:.3f}"
    print(f"{r.name:16s} {pooled:>15s} {r.median_pnl:13,.0f} "
          f"{r.wins:5d}/{len(r.pnls)}")

agent             pooled capture    median P&L  first on
mean_reversion             0.490        26,213     5/6
random                    -0.081        -5,403     0/6
mine                      -0.137        -9,509     0/6
momentum                  -0.137        -9,509     1/6
buy_and_hold              -0.308       -19,917     0/6


`pooled_capture`, total P&L over the reference's total, is the number to
quote. The `first on` column counts seeds where an entrant finished top of
the table, with `oracle` held out of the running; it is a league position
rather than a head-to-head record. Ties break on name rather than randomly,
so `momentum` takes both of the seeds `mean_reversion` did not
win, while `mine` -- identical to it in every other column -- takes none.

## The paired sign test

`decisive` is true only when one entrant won on every paired seed. That is
the strongest claim a sign test can make, and it needs no distributional
assumption.

In [7]:
for other in ("buy_and_hold", "random", "mean_reversion"):
    t = ranking.separation("mine", other)
    p = "n/a" if t["p_value"] is None else f"{t['p_value']:.3f}"
    print(f"  mine vs {other:16s} {t['wins_a']}-{t['wins_b']}"
          f"  ties {t['ties']}  decisive={t['decisive']}  p={p}")

  mine vs buy_and_hold     5-1  ties 0  decisive=False  p=0.219
  mine vs random           2-4  ties 0  decisive=False  p=0.688
  mine vs mean_reversion   1-5  ties 0  decisive=False  p=0.219


`unmeasurable` lists the seeds where capture could not be measured, because
the reference did not make money there and a ratio against a negative
denominator would flip the sign of the whole table. They are reported rather
than dropped: a result averaged over the seeds that happened to work, while
presenting itself as covering all of them, is the quiet omission this
library exists to avoid. Here nothing was dropped.

In [8]:
print("unmeasurable:", list(ranking.unmeasurable) or "none")
print()
print(ranking.report())

unmeasurable: none

6 seeds on universe 5d8de78b55aa... under model pt-v14
  mean_reversion    capture +0.490  per-seed [+0.019, +1.043]  wins 5/6
  random            capture -0.081  per-seed [-0.145, -0.036]  wins 0/6
  mine              capture -0.137  per-seed [-0.461, +0.105]  wins 0/6
  momentum          capture -0.137  per-seed [-0.461, +0.105]  wins 1/6
  buy_and_hold      capture -0.308  per-seed [-0.607, +0.057]  wins 0/6


## A caveat

Good results here do not predict real returns. The price process comes from
a known model, so a strategy that fits its structure will look excellent
without telling you anything transferable. A strategy that fails here is
more informative: it broke against a live order book under honest impact
costs.

Next: **[3 · Why did the price move](03-why-did-the-price-move.ipynb)**.